##### **** These pip installs need to be adapted to use the appropriate release level. Alternatively, The venv running the jupyter lab could be pre-configured with a requirement file that includes the right release. Example for transform developers working from git clone:
```
make venv 
source venv/bin/activate 
pip install jupyterlab
```

In [47]:
%%capture
!pip install pandas

In [8]:
import os
os.environ['S3_ACCESS_KEY']=os.environ['COS_DPK_ACCESS_KEY']
os.environ['S3_SECRET_KEY']=os.environ['COS_DPK_SECRET_KEY']
os.environ['S3_ENDPOINT']= "https://s3.us-south.cloud-object-storage.appdomain.cloud"
input_folder="S3://bucket-wikimedia-1/take6_md"
output_folder="S3://bucket-wikimedia-1/test-MT"

##### ***** Import required classes and modules

In [9]:
from data_processing.data_access import DataAccessS3

In [89]:
da=DataAccessS3(config={'input_folder': input_folder,'output_folder': output_folder})

In [90]:
l, _, _=da.get_files_to_process()


In [104]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

dataset=None
ndx = 0
for _filename in l:
    byte_array,_=da.get_file(_filename)
    ndx = ndx + 1
    print (f"{ndx}- Processing {_filename}: file size: {len(byte_array)}")
    pqfile=pq.ParquetFile(pa.BufferReader(byte_array))
    patable=pqfile.read().select(['filename', 'document_hash'])
    if dataset is None:
        dataset=patable.to_pandas()
    else:
        dataset=pd.concat([dataset, patable.to_pandas()], ignore_index=True)


1- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_0.ndjson.parquet: file size: 35053613
2- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_1.ndjson.parquet: file size: 35506589
3- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_10.ndjson.parquet: file size: 34491630
4- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_100.ndjson.parquet: file size: 50950761
5- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_101.ndjson.parquet: file size: 52202646
6- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_102.ndjson.parquet: file size: 52989262
7- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_103.ndjson.parquet: file size: 53085169
8- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_104.ndjson.parquet: file size: 53694756
9- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_105.ndjson.parquet: file size: 53821285
10- Processing bucket-wikimedia-1/take6_md/enwiki_namespace_0_106.ndjson.parquet: file size: 531

In [105]:
len(dataset)

7128273

In [120]:
hash_counts=dataset['document_hash'].value_counts()
dataset['hash_count']=dataset['document_hash'].map(hash_counts)
dataset.sort_values(by='hash_count', ascending=False)

,filename,document_hash,hash_count
5391239,enwiki_namespace_0_396_1494.html,4645722594882866820,3
5391245,enwiki_namespace_0_396_1500.html,4645722594882866820,3
5391234,enwiki_namespace_0_396_1489.html,4645722594882866820,3
5391425,enwiki_namespace_0_396_1680.html,12771608352772316580,2
5383791,enwiki_namespace_0_395_909.html,10600707456625690885,2
...,...,...,...
2376087,enwiki_namespace_0_185_7325.html,3907772245686668795,1
2376086,enwiki_namespace_0_185_7324.html,12870594153745355973,1
2376085,enwiki_namespace_0_185_7323.html,2111910482168064541,1
2376084,enwiki_namespace_0_185_7322.html,13341779551056146637,1


In [114]:
duplicates=dataset[dataset['hash_count'] == 3]
duplicates.sort_values(by='document_hash', ascending=False)

,filename,document_hash,hash_count
5391234,enwiki_namespace_0_396_1489.html,4645722594882866820,3
5391239,enwiki_namespace_0_396_1494.html,4645722594882866820,3
5391245,enwiki_namespace_0_396_1500.html,4645722594882866820,3


In [119]:
duplicates=dataset[dataset['hash_count'] == 2]
pd.set_option('display.max_rows', len(duplicates))
duplicates.sort_values(by='document_hash', ascending=False)


,filename,document_hash,hash_count
5376197,enwiki_namespace_0_394_247.html,9944271733273349128,2
5376196,enwiki_namespace_0_394_246.html,9944271733273349128,2
5310127,enwiki_namespace_0_387_4763.html,9669684869341135938,2
5310134,enwiki_namespace_0_387_4770.html,9669684869341135938,2
5280400,enwiki_namespace_0_383_1597.html,9656007629735751019,2
5280401,enwiki_namespace_0_383_1598.html,9656007629735751019,2
5468474,enwiki_namespace_0_402_5906.html,9617237508520696951,2
5468475,enwiki_namespace_0_402_5907.html,9617237508520696951,2
5381572,enwiki_namespace_0_394_5624.html,9609734338291521026,2
5381577,enwiki_namespace_0_394_5629.html,9609734338291521026,2


In [121]:
pqfile.read()

pyarrow.Table
filename: string
contents: string
num_pages: int64
num_tables: int64
num_doc_elements: int64
document_id: string
document_hash: string
ext: string
hash: string
size: int64
date_acquired: string
document_convert_time: double
source_filename: string
----
filename: [["enwiki_namespace_0_99_0.html","enwiki_namespace_0_99_1.html","enwiki_namespace_0_99_2.html","enwiki_namespace_0_99_3.html","enwiki_namespace_0_99_4.html",...,"enwiki_namespace_0_99_24186.html","enwiki_namespace_0_99_24187.html","enwiki_namespace_0_99_24188.html","enwiki_namespace_0_99_24189.html","enwiki_namespace_0_99_24190.html"]]
contents: [["# Stenoma gemellata

Species of moth

<!-- image -->

| Stenoma gemellata               | Stenoma gemellata               |
|---------------------------------|---------------------------------|
| Scientific classification       | Scientific classification       |
| Domain:                         | Eukaryota                       |
| Kingdom:                        | An